In [1]:
# Komórka 1: importy i parametry globalne

import os
import sys
import time
import math
import re
import csv
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.interpolate import griddata

# --- Wymuś UTF-8 dla stdout/stderr ---
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')
os.environ['PYTHONIOENCODING'] = 'utf-8'

# =====================================================================
# ŚCIEŻKI I ZAKRES OBRAZKÓW
# =====================================================================
path = './'
i_min, i_max = 1, 239      # wszystkie klatki

# =====================================================================
# PODKATALOGI NA PLIKI WYNIKOWE
# =====================================================================
PATH_TXT = path + 'TXT/'
PATH_PNG = path + 'PNG/'
PATH_CSV = path + 'CSV/'

os.makedirs(PATH_TXT, exist_ok=True)
os.makedirs(PATH_PNG, exist_ok=True)
os.makedirs(PATH_CSV, exist_ok=True)

# =====================================================================
# ORYGINALNY OBRAZ (klatka_N.png) I OBCIĘCIE CZARNYCH PASKÓW
# =====================================================================
y_w_orig, x_w_orig = 652, 640
Y_CUT_TOP = 6
Y_CUT_BOT = 8

y_w = y_w_orig - Y_CUT_TOP - Y_CUT_BOT    # = 638
x_w = x_w_orig                             # = 640

# =====================================================================
# ROI (OKNO PŁYWAJĄCE) — 220 × 220 px
# =====================================================================
y_d, x_d = 220, 220

# =====================================================================
# PARAMETRY CZYSZCZENIA I KWANTYZACJI
# =====================================================================
r_min = 8
r_min_detekcji = 2
r_max_detekcji = 8
DIST_SCALANIE = 10

# =====================================================================
# DELTA — ZAPAS PASKÓW DLA TORUSA
# =====================================================================
delta = 3 * r_min         # = 24

# =====================================================================
# ROZMIAR TORUSA
# =====================================================================
y_w_torus = y_w + 2 * delta    # = 686
x_w_torus = x_w + 2 * delta    # = 688

# =====================================================================
# OKNA PŁYWAJĄCE
# =====================================================================
LICZBA_OKIEN_Y = 10
LICZBA_OKIEN_X = 10

y_0_min = 0
x_0_min = 0
Y_0_RANGE = y_w - y_d       # = 418
X_0_RANGE = x_w - x_d       # = 420

krok_y = Y_0_RANGE // max(1, LICZBA_OKIEN_Y - 1)    # = 46
krok_x = X_0_RANGE // max(1, LICZBA_OKIEN_X - 1)    # = 46

y_0_max = y_0_min + (LICZBA_OKIEN_Y - 1) * krok_y   # = 414
x_0_max = x_0_min + (LICZBA_OKIEN_X - 1) * krok_x   # = 414

nr_min, nr_max = 0, LICZBA_OKIEN_Y - 1
mr_min, mr_max = 0, LICZBA_OKIEN_X - 1

assert y_0_max + y_d <= y_w
assert x_0_max + x_d <= x_w

# =====================================================================
# PARAMETRY DETEKTORÓW
# =====================================================================
BLOB_MIN_SIGMA = 2
BLOB_MAX_SIGMA = 6
BLOB_NUM_SIGMA = 5
BLOB_THRESHOLD = 0.1

# =====================================================================
# PNG / TXT / CSV
# =====================================================================
SAVE_PNG_FOR_I = 1
PNG_PREFIX = 'real'
TXT_PREFIX = 'real'
SAVE_KLATKA_BEZ_PASKOW = True
SAVE_KLATKA_TORUS = True

# =====================================================================
# DEBUG
# =====================================================================
DEBUG_NR = nr_max
DEBUG_MR = mr_max

# =====================================================================
# LIMITY BEZPIECZEŃSTWA
# =====================================================================
MAX_PUNKTOW_E2 = 500
BLOCK_SIZE_E2 = 256

# =====================================================================
# WIZUALIZACJE
# =====================================================================
VIEW_ANGLES = [(30, 45), (60, 120), (90, 180), (15, 270)]
COLUMNS_CSV = ["N", "y", "x", "Re(e2)", "Im(e2)", "|e2|", "e22"]
WIELKOSCI = [
    ("e22",    "e22",    "e22"),
    ("Re(e2)", "Re_e2",  "Re(e2)"),
    ("Im(e2)", "Im_e2",  "Im(e2)"),
    ("|e2|",   "abs_e2", "|e2|"),
]
GRID_N = 100
WIELKOSCI_STAT = ["Re(e2)", "Im(e2)", "|e2|", "e22"]

# =====================================================================
# FUNKCJA POMOCNICZA — diagnostyka pamięci GPU
# =====================================================================
def gpu_mem_info(tag=""):
    try:
        import cupy as cp
        pool = cp.get_default_memory_pool()
        used = pool.used_bytes() / 1e9
        total = pool.total_bytes() / 1e9
        print(f"[GPU {tag}] used={used:.2f} GB, total={total:.2f} GB")
    except Exception as e:
        print(f"[GPU {tag}] brak info: {e}")

print(f"=== PARAMETRY STARTOWE ===")
print(f"Obrazy: klatka_{{i:03d}}.png, i ∈ [{i_min}, {i_max}]")
print(f"Oryginalny obraz:  {y_w_orig} × {x_w_orig} (y × x)")
print(f"Obcięcie:          {Y_CUT_TOP} px z góry, {Y_CUT_BOT} px z dołu")
print(f"Po obcięciu:       {y_w} × {x_w} (y × x)")
print(f"Delta:             {delta} px (= 3 × r_min = 3 × {r_min})")
print(f"Torus:             {y_w_torus} × {x_w_torus} (y × x)")
print(f"ROI (okno):        {y_d} × {x_d} (y × x)")
print(f"Liczba okien:      {LICZBA_OKIEN_Y} × {LICZBA_OKIEN_X} = "
      f"{LICZBA_OKIEN_Y * LICZBA_OKIEN_X}")
print(f"Zakres y_0:        [{y_0_min}, {y_0_max}], krok = {krok_y}")
print(f"Zakres x_0:        [{x_0_min}, {x_0_max}], krok = {krok_x}")
print(f"Detektory:         sigma ∈ [{BLOB_MIN_SIGMA}, {BLOB_MAX_SIGMA}], "
      f"threshold = {BLOB_THRESHOLD}")
print(f"Katalogi wynikowe:")
print(f"  TXT: {PATH_TXT}")
print(f"  PNG: {PATH_PNG}")
print(f"  CSV: {PATH_CSV}")
print(f"==========================")

=== PARAMETRY STARTOWE ===
Obrazy: klatka_{i:03d}.png, i ∈ [1, 239]
Oryginalny obraz:  652 × 640 (y × x)
Obcięcie:          6 px z góry, 8 px z dołu
Po obcięciu:       638 × 640 (y × x)
Delta:             24 px (= 3 × r_min = 3 × 8)
Torus:             686 × 688 (y × x)
ROI (okno):        220 × 220 (y × x)
Liczba okien:      10 × 10 = 100
Zakres y_0:        [0, 414], krok = 46
Zakres x_0:        [0, 414], krok = 46
Detektory:         sigma ∈ [2, 6], threshold = 0.1
Katalogi wynikowe:
  TXT: ./TXT/
  PNG: ./PNG/
  CSV: ./CSV/


In [2]:
# Komórka 2: funkcje pomocnicze

def check_kdtree(array, label=""):
    """Diagnostyka: znajduje dwa najbliższe punkty i wypisuje ich odległość."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 2:
        return len(array)
    tree = cKDTree(array[:, :2])
    d, idx = tree.query(array[:, :2], k=2)
    min_dist = d[:, 1].min()
    argmin = int(d[:, 1].argmin())
    print(f"[{label}] Najbliższe: {array[argmin, :2]} i {array[idx[argmin, 1], :2]}")
    print(f"[{label}] Odległość: {min_dist:.4f}, N={len(array)}")
    return len(array)


def clean_kdtree(array, dist, label=""):
    """Scala punkty bliższe niż `dist` w jeden punkt (płaska metryka)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 2:
        return array
    tree = cKDTree(array[:, :2])
    pairs = tree.query_pairs(r=dist, output_type='ndarray')
    if len(pairs) == 0:
        print(f"[{label}] Brak par < {dist}. N={len(array)}")
        return array

    parent = np.arange(len(array))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for a, b in pairs:
        union(int(a), int(b))

    from collections import defaultdict
    klastry = defaultdict(list)
    for idx in range(len(array)):
        klastry[find(idx)].append(idx)

    wynik = np.array([array[l].mean(axis=0) for l in klastry.values()])
    print(f"[{label}] clean_kdtree: {len(array)} → {len(wynik)}")
    return wynik


def clean_kdtree_torus(array, y_0, x_0, y_d, x_d, dist, label=""):
    """Scalanie punktów z METRYKĄ TORUSOWĄ (9 kopii + etykiety)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 2:
        return array

    shifts = [
        (0, 0), (0, +x_d), (0, -x_d),
        (+y_d, 0), (-y_d, 0),
        (+y_d, +x_d), (+y_d, -x_d),
        (-y_d, +x_d), (-y_d, -x_d),
    ]
    kopie, etykiety = [], []
    for k, (dy, dx) in enumerate(shifts):
        kopia = array[:, :2].copy()
        kopia[:, 0] += dy
        kopia[:, 1] += dx
        kopia[:, 0] = y_0 + (kopia[:, 0] - y_0) % y_d
        kopia[:, 1] = x_0 + (kopia[:, 1] - x_0) % x_d
        kopie.append(kopia)
        etykiety.append(np.full(len(kopia), k, dtype=np.int32))
    kopie = np.concatenate(kopie)
    etykiety = np.concatenate(etykiety)

    tree = cKDTree(kopie)
    pairs = tree.query_pairs(r=dist, output_type='ndarray')
    if len(pairs) == 0:
        print(f"[{label}] Brak par < {dist} (torus). N={len(array)}")
        return array

    parent = np.arange(len(kopie))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    for a, b in pairs:
        union(int(a), int(b))

    from collections import defaultdict
    klastry = defaultdict(list)
    for idx in range(len(kopie)):
        klastry[find(idx)].append(idx)

    wynik = []
    for idx_lista in klastry.values():
        oryginaly = [i for i in idx_lista if etykiety[i] == 0]
        if len(oryginaly) > 0:
            sr = kopie[oryginaly].mean(axis=0)
            wynik.append(sr)

    wynik = np.array(wynik)
    print(f"[{label}] clean_kdtree_torus: {len(array)} → {len(wynik)}")
    return wynik


def kwantyzuj_do_siatki(array, y_0, x_0, y_d, x_d, krok):
    """Kwantyzacja punktów do regularnej siatki o kroku `krok` (px)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) < 1:
        return np.zeros((0, 2))
    i_idx = np.floor((array[:, 0] - y_0) / krok).astype(np.int64)
    j_idx = np.floor((array[:, 1] - x_0) / krok).astype(np.int64)

    from collections import defaultdict
    grupy = defaultdict(list)
    for k in range(len(array)):
        grupy[(i_idx[k], j_idx[k])].append(k)

    wynik = []
    for (i, j), idx_lista in grupy.items():
        sr = array[idx_lista].mean(axis=0)
        wynik.append(sr)

    wynik = np.array(wynik)
    print(f"[kwantyzuj] krok={krok:.3f}: {len(array)} → {len(wynik)}")
    return wynik


def replikuj_9x(array, y_d, x_d):
    """9 kopii punktów (bez zawijania)."""
    array = np.asarray(array, dtype=np.float64)
    if len(array) == 0:
        return np.zeros((0, 2)), np.zeros(0, dtype=np.int32)
    shifts = [
        (0, 0), (0, +x_d), (0, -x_d),
        (+y_d, 0), (-y_d, 0),
        (+y_d, +x_d), (+y_d, -x_d),
        (-y_d, +x_d), (-y_d, -x_d),
    ]
    kopie, etykiety = [], []
    for k, (dy, dx) in enumerate(shifts):
        kopia = array[:, :2].copy()
        kopia[:, 0] += dy
        kopia[:, 1] += dx
        kopie.append(kopia)
        etykiety.append(np.full(len(kopia), k, dtype=np.int32))
    return np.concatenate(kopie), np.concatenate(etykiety)


def symetryzuj(blobs_9x, y_0, x_0, y_d, x_d, dist=None):
    """Filtracja punktów z ROI + opcjonalna kwantyzacja (płaska metryka)."""
    blobs_9x = np.asarray(blobs_9x, dtype=np.float64)
    if len(blobs_9x) == 0:
        return np.zeros((0, 2))
    mask_roi = (
        (blobs_9x[:, 0] >= y_0) & (blobs_9x[:, 0] < y_0 + y_d) &
        (blobs_9x[:, 1] >= x_0) & (blobs_9x[:, 1] < x_0 + x_d)
    )
    roi = blobs_9x[mask_roi][:, :2].copy()
    if len(roi) == 0:
        return np.zeros((0, 2))
    if dist is not None and dist > 0 and len(roi) > 1:
        roi = kwantyzuj_do_siatki(roi, y_0, x_0, y_d, x_d, krok=dist)
    roi[:, 0] -= y_0
    roi[:, 1] -= x_0
    return roi

In [3]:
# Komórka 3a: I/O + wizualizacje + statystyki

import re

RE_LINE = re.compile(
    r'N=(\d+),\s*okno:\s*(\d+),(\d+),\s*e2=\((.+?)\),\s*e22=([\d.eE+-]+)'
)
RE_IMAG = re.compile(r'([-+])\s*([\d.]+(?:[eE][-+]?\d+)?)j$')


def _zapisz_wspolrzedne(blobs_roi, i, nr, mr, png_prefix='real'):
    """Zapisuje współrzędne punktów (Z NORMALIZACJĄ do [0, 1]) do TXT i CSV."""
    import numpy as np

    N = len(blobs_roi)
    txt_file = PATH_TXT + f'{png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.txt'
    csv_file = PATH_CSV + f'{png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.csv'

    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write(f"# Współrzędne punktów (Z NORMALIZACJĄ do [0, 1]) dla "
                f"i={i}, nr={nr}, mr={mr}, N={N}\n")
        f.write(f"# Układ: lokalny ROI, znormalizowany przez y_d={y_d}, x_d={x_d}\n")
        f.write(f"# Liczba punktów: {N}\n")
        f.write(f"# Format: x_norm  y_norm   (obie w [0, 1])\n")
        f.write("#" + "-" * 40 + "\n")
        for p in blobs_roi:
            yb, xb = p[0], p[1]
            x_norm = xb / x_d
            y_norm = yb / y_d
            f.write(f"{x_norm:>12.6f}  {y_norm:>12.6f}\n")

    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['x_norm', 'y_norm'])
        for p in blobs_roi:
            yb, xb = p[0], p[1]
            x_norm = xb / x_d
            y_norm = yb / y_d
            writer.writerow([f"{x_norm:.6f}", f"{y_norm:.6f}"])


def _zapisz_png(image, blobs_roi, i, nr, mr, png_prefix='real'):
    """Zapisuje PNG 1:1 (220×220) + TXT + CSV per okno."""
    import os
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np

    N = len(blobs_roi)
    fname = PATH_PNG + f'{png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.png'
    if os.path.exists(fname):
        print(f"[WIZ] UWAGA: {fname} już istnieje — nadpisuję!")

    y_max, x_max = image.shape[0], image.shape[1]

    DPI = 80
    fig = plt.figure(figsize=(x_max / DPI, y_max / DPI), dpi=DPI)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()

    ax.imshow(image, cmap='gray', origin='upper',
              extent=[0, x_max, y_max, 0],
              interpolation='nearest',
              aspect='auto')

    if len(blobs_roi) > 0:
        mask = (
            (blobs_roi[:, 0] >= 0) & (blobs_roi[:, 0] < y_max) &
            (blobs_roi[:, 1] >= 0) & (blobs_roi[:, 1] < x_max)
        )
        blobs_ok = blobs_roi[mask]
        n_poza = len(blobs_roi) - len(blobs_ok)
        if n_poza > 0:
            print(f"[WIZ] i={i} nr={nr} mr={mr}: {n_poza} punktów poza ROI — obcinam.")
        ax.scatter(blobs_ok[:, 1], blobs_ok[:, 0],
                   s=8, c='red', marker='o', linewidths=0, zorder=3)

    ax.set_xlim(0, x_max)
    ax.set_ylim(y_max, 0)

    try:
        fig.savefig(fname, dpi=DPI)
        plt.close(fig)
        print(f"[WIZ] Zapisano: {fname}  ({x_max}×{y_max} px, 1:1)")
    except Exception as e:
        print(f"[WIZ] Błąd zapisu PNG i={i} nr={nr} mr={mr}: {e}")
        try:
            plt.close('all')
        except Exception:
            pass

    try:
        _zapisz_wspolrzedne(blobs_roi, i, nr, mr, png_prefix)
        print(f"[WSP] Zapisano: {png_prefix}_i{i}_nr{nr}_mr{mr}_{N}.txt i .csv "
              f"(N={N}, kolumny x_norm, y_norm)")
    except Exception as e:
        print(f"[WSP] Błąd zapisu współrzędnych i={i} nr={nr} mr={mr}: {e}")


def konwertuj_txt_na_csv(txt_file, csv_file):
    """Konwertuje realN.txt → realN.csv."""
    if not os.path.exists(txt_file):
        return None

    n_wierszy = 0
    with open(txt_file, 'r', encoding='utf-8') as f_in, \
         open(csv_file, 'w', newline='', encoding='utf-8') as f_out:
        writer = csv.writer(f_out)
        writer.writerow(COLUMNS_CSV)

        for line in f_in:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            m = RE_LINE.match(line)
            if not m:
                continue

            N, y, x, e2_raw, e22 = m.groups()
            e2_clean = e2_raw.strip()
            match_c = RE_IMAG.search(e2_clean)
            if not match_c:
                continue
            sign = match_c.group(1)
            im_part = match_c.group(2)
            re_part = e2_clean[:match_c.start()].strip()
            im_value = ('-' if sign == '-' else '') + im_part

            try:
                re_float = float(re_part)
                im_float = float(im_value)
                e22_float = float(e22)
            except ValueError:
                continue

            modul = math.sqrt(re_float**2 + im_float**2)
            writer.writerow([N, y, x, re_part, im_value, modul, e22])
            n_wierszy += 1

    return n_wierszy


def wycinek_torus(img_torus, y_0, x_0, y_d, x_d):
    """Wycinek z torusa z ZAWIJANIEM modulo (H_t, W_t)."""
    import numpy as np
    H_t, W_t = img_torus.shape
    y_idx = (np.arange(y_0, y_0 + y_d) % H_t).astype(np.int64)
    x_idx = (np.arange(x_0, x_0 + x_d) % W_t).astype(np.int64)
    roi = img_torus[np.ix_(y_idx, x_idx)]
    return roi


def _wycinek_z_uzupelnieniem(img, y_0, x_0, y_d, x_d, fill_value=0):
    """Wycinek z obrazu `img` z uzupełnieniem czernią poza granicami."""
    import numpy as np
    H, W = img.shape
    roi = np.full((y_d, x_d), fill_value, dtype=img.dtype)

    y_src_start = max(0, y_0)
    y_src_end = min(H, y_0 + y_d)
    x_src_start = max(0, x_0)
    x_src_end = min(W, x_0 + x_d)

    if y_src_end <= y_src_start or x_src_end <= x_src_start:
        return roi

    y_dst_start = y_src_start - y_0
    y_dst_end = y_src_end - y_0
    x_dst_start = x_src_start - x_0
    x_dst_end = x_src_end - x_0

    roi[y_dst_start:y_dst_end, x_dst_start:x_dst_end] = \
        img[y_src_start:y_src_end, x_src_start:x_src_end]

    return roi


def _plot_surface_with_contours(df, z_col, z_label):
    """Rysuje figurę 2×2 z 4 wykresami 3D."""
    x = df["x"].to_numpy()
    y = df["y"].to_numpy()
    z = df[z_col].to_numpy()

    X_grid, Y_grid = np.meshgrid(
        np.linspace(min(x), max(x), GRID_N),
        np.linspace(min(y), max(y), GRID_N)
    )
    Z_grid = griddata((x, y), z, (X_grid, Y_grid), method='linear')

    fig, axes = plt.subplots(2, 2, figsize=(16, 12),
                             subplot_kw={'projection': '3d'})

    for ax, (elev, azim) in zip(axes.ravel(), VIEW_ANGLES):
        ax.scatter(x, y, z, c=z, cmap="viridis", marker="o", label="Dane")
        ax.plot_surface(X_grid, Y_grid, Z_grid, cmap="viridis", alpha=0.8)
        ax.contour3D(X_grid, Y_grid, Z_grid, 50, cmap='viridis')
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel(z_label)

        x_ticks = np.arange(min(x), max(x) + 1, 20)
        y_ticks = np.arange(min(y), max(y) + 1, 20)
        ax.set_xticks(x_ticks)
        ax.set_yticks(y_ticks)
        ax.set_xticklabels([str(int(i)) if i % 40 == 0 else "" for i in x_ticks])
        ax.set_yticklabels([str(int(i)) if i % 40 == 0 else "" for i in y_ticks])

        ax.view_init(elev, azim)
        ax.set_title(f"{z_label}  |  View: Elev={elev}°, Azim={azim}°")

    fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05,
                        wspace=0.2, hspace=0.3)
    return fig


def _plot_contour_2d(df, z_col, z_label):
    """Rysuje figurę 2D z konturami."""
    x = df["x"].to_numpy()
    y = df["y"].to_numpy()
    z = df[z_col].to_numpy()

    X_grid, Y_grid = np.meshgrid(
        np.linspace(min(x), max(x), GRID_N),
        np.linspace(min(y), max(y), GRID_N)
    )
    Z_grid = griddata((x, y), z, (X_grid, Y_grid), method='linear')

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    contourf = ax.contourf(X_grid, Y_grid, Z_grid, levels=30, cmap='viridis')
    contour = ax.contour(X_grid, Y_grid, Z_grid, levels=15,
                         colors='black', linewidths=0.5, alpha=0.6)
    ax.clabel(contour, inline=True, fontsize=7, fmt='%.2g')
    ax.scatter(x, y, c='red', s=8, marker='o', linewidths=0,
               zorder=3, label='Dane')

    cbar = fig.colorbar(contourf, ax=ax, shrink=0.8)
    cbar.set_label(z_label)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"ContourPlot: {z_label}")
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    return fig


def _wczytaj_csv(csv_file):
    """Wczytuje realN.csv do DataFrame (do wizualizacji)."""
    if not os.path.exists(csv_file):
        return None
    try:
        df = pd.read_csv(csv_file, delimiter=",", decimal=".",
                         comment="#", header=0, encoding='utf-8')
        if len(df.columns) == len(COLUMNS_CSV):
            df.columns = COLUMNS_CSV
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        potrzebne = ["x", "y", "e22", "Re(e2)", "Im(e2)", "|e2|"]
        for c in potrzebne:
            if c not in df.columns:
                return None
        df = df.dropna(subset=potrzebne)
        if len(df) < 4:
            return None
        return df
    except Exception as e:
        print(f"[WARN] Błąd wczytywania {csv_file}: {e}")
        return None


def zapisz_wizualizacje(csv_file, i):
    """Zapisuje 4×3D + 4×2D PNG do PATH_PNG."""
    df = _wczytaj_csv(csv_file)
    if df is None:
        return 0, 0, []

    n_3d, n_2d = 0, 0
    bledy = []
    for z_col, suffix, z_label in WIELKOSCI:
        png_3d = PATH_PNG + f'real{i}_3D_{suffix}.png'
        try:
            fig = _plot_surface_with_contours(df, z_col, z_label)
            fig.savefig(png_3d, dpi=100)
            plt.close(fig)
            n_3d += 1
        except Exception as e:
            bledy.append((i, f"3D {z_label}", str(e)))
            try:
                plt.close('all')
            except Exception:
                pass

        png_2d = PATH_PNG + f'real{i}_2D_{suffix}.png'
        try:
            fig = _plot_contour_2d(df, z_col, z_label)
            fig.savefig(png_2d, dpi=100)
            plt.close(fig)
            n_2d += 1
        except Exception as e:
            bledy.append((i, f"2D {z_label}", str(e)))
            try:
                plt.close('all')
            except Exception:
                pass

    return n_3d, n_2d, bledy


def wczytaj_csv_stat(csv_file):
    """Wczytuje realN.csv do statystyk."""
    if not os.path.exists(csv_file):
        return None
    try:
        df = pd.read_csv(csv_file, delimiter=",", decimal=".",
                         comment="#", header=0, encoding='utf-8')
        if len(df.columns) == len(COLUMNS_CSV):
            df.columns = COLUMNS_CSV
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        potrzebne = ["x", "y"] + WIELKOSCI_STAT
        for c in potrzebne:
            if c not in df.columns:
                print(f"[WARN] {csv_file}: brak kolumny {c}")
                return None
        df = df.dropna(subset=potrzebne)
        if len(df) < 1:
            return None
        return df
    except Exception as e:
        print(f"[WARN] Błąd wczytywania {csv_file}: {e}")
        return None


def statystyki(df):
    """Statystyki: mean, std, median, percentyle, IQR."""
    wynik = {}
    for c in WIELKOSCI_STAT:
        v = df[c].to_numpy(dtype=np.float64)
        wynik[c] = {
            'mean':      float(np.mean(v)),
            'std':       float(np.std(v, ddof=0)),
            'std_ddof1': float(np.std(v, ddof=1)) if len(v) > 1 else 0.0,
            'min':       float(np.min(v)),
            'max':       float(np.max(v)),
            'median':    float(np.median(v)),
            'q05':       float(np.percentile(v, 5)),
            'q25':       float(np.percentile(v, 25)),
            'q75':       float(np.percentile(v, 75)),
            'q95':       float(np.percentile(v, 95)),
            'iqr':       float(np.percentile(v, 75) - np.percentile(v, 25)),
            'n':         int(len(v)),
        }
    return wynik


def zapisz_resultN(result_file, i, stats, n_wierszy, y_0_zakres, x_0_zakres):
    """Zapisuje statystyki do resultN.txt."""
    with open(result_file, 'w', encoding='utf-8') as f:
        f.write(f"# Statystyki dla obrazka i={i}\n")
        f.write(f"# Źródło: real{i}.csv\n")
        f.write(f"# Liczba wierszy (okien): {n_wierszy}\n")
        f.write(f"# Zakres okien: y_0 ∈ {y_0_zakres}, x_0 ∈ {x_0_zakres}\n")
        f.write(f"# Format: wielkosc mean std std_ddof1 min max median "
                f"q05 q25 q75 q95 iqr n\n")
        f.write("#" + "-" * 130 + "\n")
        for c in WIELKOSCI_STAT:
            s = stats[c]
            f.write(
                f"{c:>8s}  "
                f"mean={s['mean']:> .6e}  "
                f"std={s['std']:> .6e}  "
                f"std_ddof1={s['std_ddof1']:> .6e}  "
                f"min={s['min']:> .6e}  "
                f"max={s['max']:> .6e}  "
                f"median={s['median']:> .6e}  "
                f"q05={s['q05']:> .6e}  "
                f"q25={s['q25']:> .6e}  "
                f"q75={s['q75']:> .6e}  "
                f"q95={s['q95']:> .6e}  "
                f"iqr={s['iqr']:> .6e}  "
                f"n={s['n']}\n"
            )


def zapisz_statystyki_dla_klatki(i):
    """Wrapper: wczytuje realN.csv, zapisuje resultN.txt."""
    csv_file = PATH_CSV + f'real{i}.csv'
    result_file = PATH_TXT + f'result{i}.txt'

    df = wczytaj_csv_stat(csv_file)
    if df is None:
        return False

    try:
        stats = statystyki(df)
        n_wierszy = len(df)
        try:
            y_0_zakres = f"[{y_0_min}, {y_0_max}], krok={krok_y}"
            x_0_zakres = f"[{x_0_min}, {x_0_max}], krok={krok_x}"
        except NameError:
            y_0_zakres = "(nieznany)"
            x_0_zakres = "(nieznany)"
        zapisz_resultN(result_file, i, stats, n_wierszy, y_0_zakres, x_0_zakres)
        print(f"[STAT] {csv_file} → {result_file}  "
              f"(N={n_wierszy}, e22: mean={stats['e22']['mean']:.4e}, "
              f"median={stats['e22']['median']:.4e})")
        return True
    except Exception as e:
        print(f"[STAT BŁĄD] {csv_file}: {e}")
        return False

In [4]:
# Komórka 3b: nowy potok torusowy
# =====================================================================
# ZMIANA: zamiast symetryzuj_torus (uśrednianie — tworzy sztuczne punkty
# w środku okna), używamy usun_duplikaty_torus (usuwa jeden z pary —
# ten z WIĘKSZYMI współrzędnymi). Zostaje punkt bliżej (0, 0).
# =====================================================================

import time


_torus_cache = {}
_blobs_cache = {}


# =====================================================================
# PRZYGOTOWANIE TORUSA
# =====================================================================

def przygotuj_torus(i):
    """Wczytuje klatka_N.png, odcina czarne paski, dokleja delta z 4 stron + rogi."""
    import cv2
    import numpy as np

    if i in _torus_cache:
        return _torus_cache[i]

    data_path = path + f'klatka_{i:03d}.png'
    img = cv2.imread(data_path, 0)
    if img is None:
        raise FileNotFoundError(f"Nie znaleziono pliku: {data_path}")

    img_bez_paskow = img[Y_CUT_TOP:y_w_orig - Y_CUT_BOT, :]

    if SAVE_KLATKA_BEZ_PASKOW:
        out_path = path + f'klatka_{i:03d}-bez_paskow.png'
        cv2.imwrite(out_path, img_bez_paskow)
        print(f"[TORUS] Zapisano: {out_path}  "
              f"({img_bez_paskow.shape[0]}×{img_bez_paskow.shape[1]} px)")

    H, W = img_bez_paskow.shape
    img_torus = np.zeros((H + 2 * delta, W + 2 * delta), dtype=img_bez_paskow.dtype)

    img_torus[delta:delta + H, delta:delta + W] = img_bez_paskow
    img_torus[delta:delta + H, delta + W:delta + W + delta] = img_bez_paskow[:, 0:delta]
    img_torus[delta:delta + H, 0:delta] = img_bez_paskow[:, W - delta:W]
    img_torus[delta + H:delta + H + delta, delta:delta + W] = img_bez_paskow[0:delta, :]
    img_torus[0:delta, delta:delta + W] = img_bez_paskow[H - delta:H, :]
    img_torus[0:delta, 0:delta] = img_bez_paskow[H - delta:H, W - delta:W]
    img_torus[0:delta, delta + W:delta + W + delta] = img_bez_paskow[H - delta:H, 0:delta]
    img_torus[delta + H:delta + H + delta, 0:delta] = img_bez_paskow[0:delta, W - delta:W]
    img_torus[delta + H:delta + H + delta, delta + W:delta + W + delta] = \
        img_bez_paskow[0:delta, 0:delta]

    if SAVE_KLATKA_TORUS:
        out_path = path + f'klatka_{i:03d}-torus.png'
        cv2.imwrite(out_path, img_torus)
        print(f"[TORUS] Zapisano: {out_path}  "
              f"({img_torus.shape[0]}×{img_torus.shape[1]} px)")

    _torus_cache[i] = (img_torus, img_bez_paskow)
    return img_torus, img_bez_paskow


# =====================================================================
# DETEKCJA BLOBÓW RAZ NA TORUSIE
# =====================================================================

def wykryj_bloby_na_torusie(i, debug=False):
    """Detekcja bakterii RAZ na torusie + odrzucenie pasków + clean torus."""
    import cv2
    import numpy as np
    from skimage.feature import blob_log, blob_dog
    from math import sqrt

    if i in _blobs_cache:
        return _blobs_cache[i]

    img_torus, img_bez_paskow = przygotuj_torus(i)
    H, W = img_bez_paskow.shape

    def denoise(image):
        thresh_val, thresh = cv2.threshold(
            image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )
        kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
        result = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
        return result, thresh_val

    img_denoised, thresh_val = denoise(img_torus)

    if debug:
        print(f"[DET] Otsu threshold = {thresh_val:.1f}")
        n_bialych = int((img_denoised > 127).sum())
        print(f"[DET] img_denoised: białych={n_bialych} "
              f"({100*n_bialych/img_denoised.size:.2f}%)")

    if SAVE_KLATKA_TORUS:
        debug_bw_path = path + f'debug_bw_i{i}_torus.png'
        cv2.imwrite(debug_bw_path, img_denoised)
        print(f"[DET] Zapisano: {debug_bw_path}")

    t0 = time.time()
    blobs_log = blob_log(img_denoised, min_sigma=BLOB_MIN_SIGMA,
                         max_sigma=BLOB_MAX_SIGMA, num_sigma=BLOB_NUM_SIGMA,
                         threshold=BLOB_THRESHOLD)
    blobs_dog = blob_dog(img_denoised, min_sigma=BLOB_MIN_SIGMA,
                         max_sigma=BLOB_MAX_SIGMA,
                         threshold=BLOB_THRESHOLD)
    if len(blobs_log) > 0:
        blobs_log[:, 2] = blobs_log[:, 2] * sqrt(2)
    if len(blobs_dog) > 0:
        blobs_dog[:, 2] = blobs_dog[:, 2] * sqrt(2)
    t_detekcja = time.time() - t0

    if debug:
        print(f"[DET] blobs_log: N={len(blobs_log)}")
        print(f"[DET] blobs_dog: N={len(blobs_dog)}")
        print(f"[DET] detekcja: {t_detekcja:.2f} s")

    def filtruj_promienie(blobs, r_min, r_max):
        if len(blobs) == 0:
            return blobs
        mask = (blobs[:, 2] >= r_min) & (blobs[:, 2] <= r_max)
        return blobs[mask]

    blobs_log_f = filtruj_promienie(blobs_log, r_min_detekcji, r_max_detekcji)
    blobs_dog_f = filtruj_promienie(blobs_dog, r_min_detekcji, r_max_detekcji)

    if len(blobs_log_f) > 0 and len(blobs_dog_f) > 0:
        blobs_sum = np.concatenate([blobs_log_f[:, :2], blobs_dog_f[:, :2]], axis=0)
    elif len(blobs_log_f) > 0:
        blobs_sum = blobs_log_f[:, :2]
    elif len(blobs_dog_f) > 0:
        blobs_sum = blobs_dog_f[:, :2]
    else:
        blobs_sum = np.zeros((0, 2))

    if len(blobs_sum) > 0:
        mask_oryginal = (
            (blobs_sum[:, 0] >= delta) & (blobs_sum[:, 0] < delta + H) &
            (blobs_sum[:, 1] >= delta) & (blobs_sum[:, 1] < delta + W)
        )
        blobs_sum = blobs_sum[mask_oryginal]

    if len(blobs_sum) > 0:
        blobs_sum = blobs_sum.copy()
        blobs_sum[:, 0] -= delta
        blobs_sum[:, 1] -= delta

    if len(blobs_sum) > 0:
        blobs_sum = clean_kdtree_torus(blobs_sum, 0, 0, y_w, x_w, r_min,
                                        label=f"clean_torus i={i}")

    if debug:
        print(f"[DET] Po odrzuceniu pasków i clean: N={len(blobs_sum)}")

    _blobs_cache[i] = blobs_sum
    return blobs_sum


# =====================================================================
# USUNIĘCIE DUPLIKATÓW TORUSOWYCH — bez uśredniania
# =====================================================================

def usun_duplikaty_torus(blobs_roi, y_d, x_d, r_min):
    """
    Usuwa duplikaty torusowe: jeśli dwa punkty są bliższe niż r_min
    w metryce torusowej, USUWA ten z pary, który ma WIĘKSZE współrzędne
    (y, x). Zostaje punkt z mniejszymi współrzędnymi (bliżej (0, 0)).

    Alternatywa dla symetryzuj_torus (które uśrednia — tworzy sztuczne punkty).

    Parametry:
        blobs_roi : (N, 2) — punkty w układzie lokalnym okna [0, y_d) × [0, x_d)
        y_d, x_d  : rozmiar okna (okres torusa)
        r_min     : próg scalania (px)

    Zwraca:
        (M, 2) — punkty po usunięciu duplikatów, M ≤ N.
    """
    import numpy as np
    from scipy.spatial import cKDTree

    blobs_roi = np.asarray(blobs_roi, dtype=np.float64)
    if len(blobs_roi) < 2:
        return blobs_roi

    N = len(blobs_roi)

    # 9 kopii punktów — z oryginalnym indeksem
    shifts = [
        (0, 0), (0, +x_d), (0, -x_d),
        (+y_d, 0), (-y_d, 0),
        (+y_d, +x_d), (+y_d, -x_d),
        (-y_d, +x_d), (-y_d, -x_d),
    ]
    kopie_lista, orig_idx_lista = [], []
    for k, (dy, dx) in enumerate(shifts):
        kopia = blobs_roi.copy()
        kopia[:, 0] += dy
        kopia[:, 1] += dx
        kopie_lista.append(kopia)
        orig_idx_lista.append(np.arange(N, dtype=np.int32))
    kopie = np.concatenate(kopie_lista)
    orig_idx_all = np.concatenate(orig_idx_lista)

    # cKDTree na 9N punktach
    tree_k = cKDTree(kopie)
    pairs = tree_k.query_pairs(r=r_min, output_type='ndarray')

    if len(pairs) == 0:
        print(f"[usun_duplikaty] Brak par < {r_min}. N={N}")
        return blobs_roi

    # Dla każdej pary — oznacz do usunięcia ten o WIĘKSZYCH współrzędnych
    do_usuniecia = np.zeros(N, dtype=bool)

    for a, b in pairs:
        orig_a = orig_idx_all[a]
        orig_b = orig_idx_all[b]

        if orig_a == orig_b:
            continue  # ten sam oryginał (kopia samego siebie)

        p_a = blobs_roi[orig_a]
        p_b = blobs_roi[orig_b]

        # Porównanie leksykograficzne (y, x)
        if (p_a[0] > p_b[0]) or (p_a[0] == p_b[0] and p_a[1] > p_b[1]):
            do_usuniecia[orig_a] = True
        else:
            do_usuniecia[orig_b] = True

    wynik = blobs_roi[~do_usuniecia]
    n_usunietych = int(do_usuniecia.sum())
    print(f"[usun_duplikaty] {N} → {len(wynik)} (usunięto {n_usunietych})")
    return wynik


# =====================================================================
# MACIERZ p_weierstrass BEZ dodatkowego π
# =====================================================================

def p_weierstrass_matrix(z, g2=189.073, g3=0, block_cpu=10_000):
    """Macierz p_weierstrass(z) BEZ dodatkowej π (diagonala = 0)."""
    import cupy as cp
    from mpmath import jtheta, exp, sqrt as msqrt, polyroots, agm
    from mpmath import mp, pi

    def p_weierstrass_from_g2_g3(g2, g3, derivative=0):
        if derivative < 0 or derivative > 3 or not isinstance(derivative, int):
            raise ValueError("`derivative` must be an integer between 0 and 3.")
        r1, r2, r3 = polyroots([4, 0, -g2, -g3])
        e3 = r3
        a = msqrt(r1 - r3)
        b = msqrt(r1 - r2)
        c = msqrt(r2 - r3)
        if abs(a + b) < abs(a - b):
            b *= -1
        if abs(a + c) < abs(a - c):
            c *= -1
        if abs(c + 1j * b) < abs(c - 1j * b):
            e3 = r1
            a = msqrt(r3 - r1)
            b = msqrt(r3 - r2)
            c = msqrt(r2 - r1)
            w1 = 1 / agm(1j * b, c)
        else:
            w1 = 1 / agm(a, b)
        w3 = 1j / agm(a, c)
        q = exp(1j * pi * w3 / w1)
        if derivative != 1:
            pw0 = lambda z: (
                e3 + (pi * jtheta(2, 0, q) * jtheta(3, 0, q) * jtheta(4, z / w1, q)
                      / (pi * w1 * jtheta(1, z / w1, q))) ** 2
            )
            if derivative == 0:
                return pw0
            if derivative == 2:
                return lambda z: 6 * pw0(z) ** 2 - g2 / 2
        f = (jtheta(1, 0, q, 1) ** 3
             / (jtheta(2, 0, q) * jtheta(3, 0, q) * jtheta(4, 0, q)))
        pw1 = lambda z: (
            -2 * (1 / w1) ** 3 * jtheta(2, z / w1, q) * jtheta(3, z / w1, q)
            * jtheta(4, z / w1, q) * f / jtheta(1, z / w1, q) ** 3
        )
        if derivative == 1:
            return pw1
        if derivative == 3:
            return lambda z: 12 * pw0(z) * pw1(z)

    wynik = cp.zeros_like(z, dtype=cp.complex128)
    z_flat = z.ravel()
    wynik_flat = wynik.ravel()
    nz_idx = cp.where(z_flat != 0)[0]

    for start in range(0, len(nz_idx), block_cpu):
        idx_block = nz_idx[start:start + block_cpu]
        z_block = z_flat[idx_block].get()
        vals = [p_weierstrass_from_g2_g3(g2, g3)(zi) for zi in z_block]
        wynik_flat[idx_block] = cp.array(vals, dtype=cp.complex128)

    return wynik


# =====================================================================
# PREPARE
# =====================================================================

def prepare(x, y, i, save_png_for_i=1, png_prefix='real',
            dist_scalanie=10, debug_nr=2, debug_mr=2):
    """Dla okna (nr=y, mr=x) klatki i: bloby + usunięcie duplikatów + e2/e22 + PNG/TXT/CSV."""
    import numpy as np
    import cupy as cp
    from mpmath import pi

    y_0 = y_0_min + y * krok_y
    x_0 = x_0_min + x * krok_x

    blobs_sum = wykryj_bloby_na_torusie(i, debug=(i == save_png_for_i and
                                                   x == debug_mr and y == debug_nr))

    if len(blobs_sum) == 0:
        print(f"Brak detekcji dla i={i}, nr={y}, mr={x} — pomijam.")
        if i == save_png_for_i:
            _, img_bez_paskow = przygotuj_torus(i)
            roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
            _zapisz_png(roi, np.zeros((0, 2)), i, y, x, png_prefix)
        return None

    mask_okno = (
        (blobs_sum[:, 0] >= y_0) & (blobs_sum[:, 0] < y_0 + y_d) &
        (blobs_sum[:, 1] >= x_0) & (blobs_sum[:, 1] < x_0 + x_d)
    )
    blobs_okno = blobs_sum[mask_okno].copy()

    if len(blobs_okno) == 0:
        print(f"Brak blobów w oknie i={i}, nr={y}, mr={x} — pomijam.")
        if i == save_png_for_i:
            _, img_bez_paskow = przygotuj_torus(i)
            roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
            _zapisz_png(roi, np.zeros((0, 2)), i, y, x, png_prefix)
        return None

    blobs_okno[:, 0] -= y_0
    blobs_okno[:, 1] -= x_0

    # --- 4. Usunięcie duplikatów torusowych (zamiast uśredniania) ---
    blobs_roi = usun_duplikaty_torus(blobs_okno, y_d, x_d, r_min)

    # Opcjonalna kwantyzacja (bez przesuwania układu — y_0 = x_0 = 0)
    if len(blobs_roi) > 1:
        blobs_roi = kwantyzuj_do_siatki(blobs_roi, 0, 0, y_d, x_d, krok=dist_scalanie)

    if len(blobs_roi) == 0:
        print(f"Po usunięciu duplikatów brak punktów i={i}, nr={y}, mr={x} — pomijam.")
        if i == save_png_for_i:
            _, img_bez_paskow = przygotuj_torus(i)
            roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
            _zapisz_png(roi, np.zeros((0, 2)), i, y, x, png_prefix)
        return None

    print(f"[DIAG] i={i} nr={y} mr={x}: sum={len(blobs_sum)}, "
          f"okno={len(blobs_okno)}, roi={len(blobs_roi)}")

    if i == save_png_for_i:
        _, img_bez_paskow = przygotuj_torus(i)
        roi = _wycinek_z_uzupelnieniem(img_bez_paskow, y_0, x_0, y_d, x_d, 0)
        _zapisz_png(roi, blobs_roi, i, y, x, png_prefix)

    # --- Normalizacja do [0, 1] + limit ---
    blobs_sum2 = blobs_roi.copy()
    blobs_sum2[:, 0] = blobs_roi[:, 0] / y_d
    blobs_sum2[:, 1] = blobs_roi[:, 1] / x_d

    N_roi = len(blobs_sum2)
    if N_roi > MAX_PUNKTOW_E2:
        print(f"[LIMIT] blobs_sum2 ma {N_roi} punktów > {MAX_PUNKTOW_E2} — losuję podzbiór.")
        idx = np.random.choice(N_roi, MAX_PUNKTOW_E2, replace=False)
        blobs_sum2 = blobs_sum2[idx]

    # --- e2 + e22 z WSPÓŁDZIELONĄ macierzą P_matrix ---
    from mpmath import mp
    mp.dps = 15

    array = cp.asarray(blobs_sum2)
    N = len(array)

    M_e2 = 1 / (N ** 2)
    p_local = 2
    M_e22 = ((-1) ** p_local) / (N ** (p_local + 1))

    a = cp.asarray(array[:, 0] + 1j * array[:, 1], dtype=cp.complex128)
    t0_e2 = time.time()

    diffs = a[:, None] - a[None, :]
    cp.fill_diagonal(diffs, cp.complex128(0))

    P_matrix = p_weierstrass_matrix(diffs, g2=189.073, g3=0, block_cpu=10_000)

    f2 = cp.sum(P_matrix)
    e2 = cp.complex128(pi()) + M_e2 * f2

    E2_matrix = P_matrix + cp.complex128(pi())
    fpp = cp.sum(E2_matrix, axis=1)
    e22 = M_e22 * cp.sum(cp.abs(fpp) ** 2)

    t_e2 = time.time() - t0_e2
    e22_val = float(cp.asnumpy(e22).real) if hasattr(e22, 'real') else float(e22)

    wynik = f"N={N}, okno: {y_0},{x_0}, e2={e2}, e22={e22_val}\n"
    print(wynik)
    gpu_mem_info(f"i={i} nr={y} mr={x} po e22")
    print(f"[TIMING] i={i} nr={y} mr={x} | e2+e22={t_e2:.2f}s | N={N}")
    return wynik


# =====================================================================
# PRZETWORZENIE JEDNEJ KLATKI
# =====================================================================

def przetworz_klatke(i):
    """Pełny potok dla klatki i."""
    t_start = time.time()
    print(f"\n{'='*70}")
    print(f"OBRAZEK i={i}  ({i - i_min + 1}/{i_max - i_min + 1})")
    print(f"{'='*70}")

    data_txt = PATH_TXT + TXT_PREFIX + str(i) + '.txt'
    data_csv = PATH_CSV + TXT_PREFIX + str(i) + '.csv'

    print(f"[KROK 1+2] Przygotowanie torusa i detekcja...")
    przygotuj_torus(i)
    wykryj_bloby_na_torusie(i, debug=True)

    with open(data_txt, "w", encoding='utf-8') as f:
        f.write(f"# Analiza obrazka i={i} (klatka_{i:03d}-torus.png)\n")
        f.write(f"# Oryginał: {y_w_orig}×{x_w_orig}, obcięcie [{Y_CUT_TOP}:"
                f"{y_w_orig - Y_CUT_BOT}], po obcięciu {y_w}×{x_w}\n")
        f.write(f"# Torus: {y_w_torus}×{x_w_torus} (delta={delta})\n")
        f.write(f"# ROI: {y_d}×{x_d}\n")
        f.write(f"# Okna: nr ∈ [{nr_min},{nr_max}] ({LICZBA_OKIEN_Y}), "
                f"mr ∈ [{mr_min},{mr_max}] ({LICZBA_OKIEN_X})\n")
        f.write(f"# y_0 ∈ [{y_0_min},{y_0_max}], krok_y={krok_y}\n")
        f.write(f"# x_0 ∈ [{x_0_min},{x_0_max}], krok_x={krok_x}\n")
        f.write(f"# r_min={r_min}, delta={delta}\n")
        f.write(f"# Kwantyzacja: DIST_SCALANIE={DIST_SCALANIE} px\n")
        f.write(f"# Detektory: sigma ∈ [{BLOB_MIN_SIGMA}, {BLOB_MAX_SIGMA}], "
                f"num_sigma={BLOB_NUM_SIGMA}, threshold={BLOB_THRESHOLD}\n")
        f.write(f"# Format: N, okno_y0,x0, e2, e22\n\n")

    n_okien_ok = 0
    n_okien_total = 0
    for nr in range(nr_max, nr_min - 1, -1):
        for mr in range(mr_max, mr_min - 1, -1):
            n_okien_total += 1
            print(f"---><--- i={i}, nr={nr}, mr={mr} ---><---")
            try:
                wynik = prepare(mr, nr, i,
                                save_png_for_i=SAVE_PNG_FOR_I,
                                png_prefix=PNG_PREFIX,
                                dist_scalanie=DIST_SCALANIE,
                                debug_nr=DEBUG_NR,
                                debug_mr=DEBUG_MR)
                if wynik is not None:
                    with open(data_txt, "a", encoding='utf-8') as f:
                        f.write(wynik)
                    n_okien_ok += 1
            except Exception as e:
                print(f"BŁĄD dla i={i}, nr={nr}, mr={mr}: {e}")
                with open(data_txt, "a", encoding='utf-8') as f:
                    f.write(f"# BŁĄD i={i}, nr={nr}, mr={mr}: {e}\n")
                try:
                    import cupy as cp
                    cp.get_default_memory_pool().free_all_blocks()
                except Exception:
                    pass

    t_txt = time.time()
    print(f"\n[KROK 3] TXT zapisany: {data_txt}  "
          f"({n_okien_ok}/{n_okien_total} okien, {t_txt - t_start:.1f} s)")

    n_wierszy = konwertuj_txt_na_csv(data_txt, data_csv)
    if n_wierszy is None:
        print(f"[KROK 4] Nie udało się utworzyć CSV dla i={i}")
        return {'i': i, 'txt': False, 'csv': False, 'png_3d': 0, 'png_2d': 0,
                'stat': False, 'czas': time.time() - t_start}

    t_csv = time.time()
    print(f"[KROK 4] CSV zapisany: {data_csv}  ({n_wierszy} wierszy, "
          f"{t_csv - t_txt:.1f} s)")

    n_3d, n_2d, bledy = zapisz_wizualizacje(data_csv, i)
    t_png = time.time()
    print(f"[KROK 5] PNG zapisane: {n_3d}×3D + {n_2d}×2D, "
          f"{t_png - t_csv:.1f} s")
    if bledy:
        print(f"[KROK 5] Błędy: {len(bledy)}")
        for j, typ, err in bledy[:3]:
            print(f"    i={j}, {typ}: {err}")

    ok_stat = zapisz_statystyki_dla_klatki(i)
    t_stat = time.time()

    czas_total = time.time() - t_start
    print(f">>> Klatka i={i} zakończona w {czas_total:.1f} s "
          f"(TXT {t_txt - t_start:.1f}s + CSV {t_csv - t_txt:.1f}s + "
          f"PNG {t_png - t_csv:.1f}s + STAT {t_stat - t_png:.1f}s)")

    return {'i': i, 'txt': True, 'csv': True,
            'png_3d': n_3d, 'png_2d': n_2d,
            'stat': ok_stat,
            'czas': czas_total}

In [ ]:
# Komórka 4: pętla główna po klatkach

t_global_start = time.time()
wyniki = []

for i in range(i_min, i_max + 1):
    try:
        wynik = przetworz_klatke(i)
        wyniki.append(wynik)
        gpu_mem_info(f"po i={i}")
    except Exception as e:
        import traceback
        print(f"\n!!! KRYTYCZNY BŁĄD dla i={i}: {e}")
        traceback.print_exc()
        wyniki.append({
            'i': i, 'txt': False, 'csv': False,
            'png_3d': 0, 'png_2d': 0,
            'stat': False, 'czas': 0
        })
        try:
            import cupy as cp
            cp.get_default_memory_pool().free_all_blocks()
        except Exception:
            pass
        try:
            _torus_cache.pop(i, None)
            _blobs_cache.pop(i, None)
        except Exception:
            pass

t_global_end = time.time()
print(f"\n{'='*70}")
print(f"PODSUMOWANIE KOŃCOWE")
print(f"{'='*70}")

n_ok = sum(1 for w in wyniki if w['txt'])
n_csv = sum(1 for w in wyniki if w['csv'])
n_stat = sum(1 for w in wyniki if w.get('stat', False))
n_3d_total = sum(w['png_3d'] for w in wyniki)
n_2d_total = sum(w['png_2d'] for w in wyniki)
czas_total = t_global_end - t_global_start

print(f"Klatek przetworzonych: {len(wyniki)} (z {i_max - i_min + 1})")
print(f"  z TXT:               {n_ok}")
print(f"  z CSV:               {n_csv}")
print(f"  z resultN.txt:       {n_stat}")
print(f"PNG 3D:                {n_3d_total}")
print(f"PNG 2D:                {n_2d_total}")
print(f"Łączny czas:           {czas_total:.1f} s ({czas_total/60:.1f} min)")

if wyniki:
    czasy = [w['czas'] for w in wyniki if w['czas'] > 0]
    if czasy:
        print(f"Średni czas na klatkę: {sum(czasy)/len(czasy):.1f} s")

pominiete = [w['i'] for w in wyniki if not w['txt']]
if pominiete:
    print(f"\nKlatek bez TXT: {len(pominiete)}")
    if len(pominiete) <= 20:
        print(f"  Numery: {pominiete}")
    else:
        print(f"  Numery: {pominiete[:10]} ... {pominiete[-10:]}")

try:
    _torus_cache.clear()
    _blobs_cache.clear()
    print(f"\n[CACHE] Wyczyszczono cache torusa i blobów.")
except Exception:
    pass

print(f"\nKatalogi wynikowe:")
try:
    print(f"  TXT: {PATH_TXT}")
    print(f"  PNG: {PATH_PNG}")
    print(f"  CSV: {PATH_CSV}")
except NameError:
    print(f"  (brak zmiennych PATH_* — uruchom Komórkę 1)")

print(f"{'='*70}")

In [ ]:
# Komórka 5: zbiorczy result_all.txt

import os
import numpy as np
import pandas as pd

try:
    _i_min, _i_max = i_min, i_max
except NameError:
    _i_min, _i_max = 1, 239
try:
    COLUMNS_CSV
except NameError:
    COLUMNS_CSV = ["N", "y", "x", "Re(e2)", "Im(e2)", "|e2|", "e22"]
try:
    WIELKOSCI_STAT
except NameError:
    WIELKOSCI_STAT = ["Re(e2)", "Im(e2)", "|e2|", "e22"]

try:
    _path_txt = PATH_TXT
    _path_csv = PATH_CSV
except NameError:
    _path = './'
    _path_txt = _path + 'TXT/'
    _path_csv = _path + 'CSV/'
    os.makedirs(_path_txt, exist_ok=True)
    os.makedirs(_path_csv, exist_ok=True)


def wczytaj_csv_stat(csv_file):
    if not os.path.exists(csv_file):
        return None
    try:
        df = pd.read_csv(csv_file, delimiter=",", decimal=".",
                         comment="#", header=0, encoding='utf-8')
        if len(df.columns) == len(COLUMNS_CSV):
            df.columns = COLUMNS_CSV
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        potrzebne = ["x", "y"] + WIELKOSCI_STAT
        for c in potrzebne:
            if c not in df.columns:
                return None
        df = df.dropna(subset=potrzebne)
        if len(df) < 1:
            return None
        return df
    except Exception as e:
        print(f"[WARN] Błąd wczytywania {csv_file}: {e}")
        return None


def statystyki(df):
    wynik = {}
    for c in WIELKOSCI_STAT:
        v = df[c].to_numpy(dtype=np.float64)
        wynik[c] = {
            'mean':   float(np.mean(v)),
            'std':    float(np.std(v, ddof=0)),
            'median': float(np.median(v)),
            'q25':    float(np.percentile(v, 25)),
            'q75':    float(np.percentile(v, 75)),
            'n':      int(len(v)),
        }
    return wynik


result_all = _path_txt + 'result_all.txt'

lacznie_plikow = 0
lacznie_wierszy = 0
pominiete = []
bledy = []

with open(result_all, 'w', encoding='utf-8') as f_all:
    f_all.write("# Zbiorcze statystyki dla wszystkich klatek\n")
    f_all.write(f"# Zakres klatek: i ∈ [{_i_min}, {_i_max}]\n")
    f_all.write("# Format: i  N  "
                "Re(e2)_mean  Re(e2)_median  Re(e2)_std  "
                "Im(e2)_mean  Im(e2)_median  Im(e2)_std  "
                "|e2|_mean  |e2|_median  |e2|_std  "
                "e22_mean  e22_median  e22_std\n")
    f_all.write("#" + "-" * 160 + "\n")

    for i in range(_i_min, _i_max + 1):
        csv_file = _path_csv + f'real{i}.csv'
        df = wczytaj_csv_stat(csv_file)
        if df is None:
            pominiete.append(i)
            continue

        try:
            stats = statystyki(df)
            n_wierszy = len(df)
            f_all.write(
                f"{i:>4d}  {n_wierszy:>4d}  "
                f"{stats['Re(e2)']['mean']:> .4e}  "
                f"{stats['Re(e2)']['median']:> .4e}  "
                f"{stats['Re(e2)']['std']:> .4e}  "
                f"{stats['Im(e2)']['mean']:> .4e}  "
                f"{stats['Im(e2)']['median']:> .4e}  "
                f"{stats['Im(e2)']['std']:> .4e}  "
                f"{stats['|e2|']['mean']:> .4e}  "
                f"{stats['|e2|']['median']:> .4e}  "
                f"{stats['|e2|']['std']:> .4e}  "
                f"{stats['e22']['mean']:> .4e}  "
                f"{stats['e22']['median']:> .4e}  "
                f"{stats['e22']['std']:> .4e}\n"
            )
            lacznie_plikow += 1
            lacznie_wierszy += n_wierszy
            print(f"[OK] real{i}.csv → result_all.txt  "
                  f"(N={n_wierszy}, e22: mean={stats['e22']['mean']:.4e}, "
                  f"median={stats['e22']['median']:.4e})")
        except Exception as e:
            print(f"[BŁĄD] real{i}.csv: {e}")
            bledy.append((i, str(e)))

print()
print(f"=== Podsumowanie Komórki 5 ===")
print(f"Przetworzonych plików:     {lacznie_plikow}")
print(f"Łącznie wierszy danych:    {lacznie_wierszy}")
print(f"Zbiorczy plik:             {result_all}")
if pominiete:
    print(f"Pominięto (brak CSV):      {len(pominiete)}")
    if len(pominiete) <= 20:
        print(f"  Numery: {pominiete}")
    else:
        print(f"  Numery: {pominiete[:10]} ... {pominiete[-10:]}")
if bledy:
    print(f"Błędy:                     {len(bledy)}")
    for i, err in bledy[:5]:
        print(f"  i={i}: {err}")